# Fly-Wind Example — JAX Accelerated

This notebook is a JAX-accelerated version of `fly_wind_example.ipynb`.
The physics, MPC simulation, and downstream analysis are identical — only
the observability matrix computation is replaced.

## What changes vs. the original notebook

| Step | Original | JAX version |
|------|----------|-------------|
| Simulation / MPC | `Simulator` (do_mpc / CasADi IDAS) | **unchanged** |
| Observability matrix | `SlidingEmpiricalObservabilityMatrix` — 2n perturbation sims per window | `JaxSlidingEmpiricalObservabilityMatrix` — one `jax.vmap(jax.jacfwd)` call for all windows |
| Fisher information | `SlidingFisherObservability` | **unchanged** |
| Plotting | same | **unchanged** |

## What you need to write differently

1. **Install JAX**: `pip install "jax[cpu]"`
2. **Write `f_jax` and `h_jax`** using `jax.numpy` instead of `numpy` for all math operations (`jnp.cos`, `jnp.sin`, `jnp.arctan2`, `jnp.sqrt`). Return `jnp.array(...)` instead of a list.
3. **Remove data-dependent control flow** from `h_jax`. The original `h` unwraps angles when given a whole trajectory array, but `JaxSimulator` evaluates `h(x, u)` one time step at a time (scalars), so unwrapping is not needed.
4. **Create a `JaxSimulator`** with the JAX functions.
5. **Replace** `SlidingEmpiricalObservabilityMatrix(simulator, ...)` with `JaxSlidingEmpiricalObservabilityMatrix(jax_simulator, ...)`.

# Install pybounds and requirements

This should allow the notebook to run directly in Google Colab.

In [ ]:
try:
    import casadi
except:
    !pip install casadi
    import casadi

try:
    import do_mpc
except:
    !pip install do_mpc
    import do_mpc

try:
    import pybounds
    from pybounds import JaxSimulator
except (ImportError, AttributeError):
    !pip install "git+https://github.com/vanbreugel-lab/pybounds@claude_explorations"
    import pybounds

try:
    import jax
except:
    !pip install "jax[cpu]"
    import jax

# Import packages

In [ ]:
import time
import numpy as np
import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt

from pybounds import (Simulator,
                      JaxSimulator, JaxSlidingEmpiricalObservabilityMatrix,
                      FisherObservability, SlidingFisherObservability,
                      ObservabilityMatrixImage, colorline)

# Define system dynamics and measurements

This example uses a model of an insect flying in the presence of wind.

See the following reference for details:

Floris van Breugel
A Nonlinear Observability Analysis of Ambient Wind Estimation with Uncalibrated Sensors, Inspired by Insect Neural Encoding
2021 60th IEEE Conference on Decision and Control (CDC)
DOI: 10.1109/CDC45484.2021.9683219

The system dynamics are described by seven primary states:
* altitude $z$
* parallel velocity $v_{\parallel}$
* perpendicular velocity $v_{\perp}$
* heading $\phi$
* angular velocity $\dot{\phi}$
* wind speed $w$
* wind direction $\zeta$

And the system dynamics are given by
$$
\dot{\mathbf{x}} = \begin{bmatrix} \dot{z} \\ \dot{v}_{\parallel} \\ \dot{v}_{\perp} \\ \dot{\phi} \\ \ddot{\phi} \\ \dot{w}  \\ \dot{\zeta} \end{bmatrix} =
f(\mathbf{x},\mathbf{u}) = \begin{bmatrix}
\dot{z} \\
\frac{1}{m}(k_{m_1}u_{\parallel} - C_{\parallel} a_{\parallel}) + v_{\perp} \dot{\phi} \\
 \frac{1}{m}(k_{m_3}u_{\perp} - C_{\perp} a_{\perp}) - v_{\parallel} \dot{\phi} \\
  \dot{\phi} \\
   \frac{1}{I}(k_{m_4}u_{\phi} - C_{\phi} \dot{\phi} + k_{m_2} u_{\perp}) \\
    \dot{w} \\
     \dot{\zeta} \\
\end{bmatrix}
$$

The putative system measurements are:
* heading $\phi$
* ground speed angle $\psi$
* apparent airflow angle $\gamma$
* apparent airflow magnitude $a$
* ground speed magnitude $g$
* optic flow $g/z$

In [ ]:
state_names = ['x', 'y', 'z', 'v_para', 'v_perp', 'phi', 'phi_dot',
               'w', 'zeta', 'm', 'I', 'C_para', 'C_perp', 'C_phi',
               'km1', 'km2', 'km3', 'km4']

input_names = ['u_para', 'u_perp', 'u_phi']

measurement_names = ['phi', 'psi', 'gamma', 'a', 'g', 'r']

dt = 0.01  # [s]

## Legacy dynamics and measurement (for `Simulator` / do_mpc)

Unchanged from the original notebook — used for MPC and trajectory simulation.

In [ ]:
def f(X, U):
    x, y, z, v_para, v_perp, phi, phi_dot, w, zeta, m, I, C_para, C_perp, C_phi, km1, km2, km3, km4 = X
    u_para, u_perp, u_phi = U

    a_para = v_para - w * np.cos(phi - zeta)
    a_perp = v_perp + w * np.sin(phi - zeta)

    v_para_dot = ((km1 * u_para - C_para * a_para) / m) + (v_perp * phi_dot)
    v_perp_dot = ((km3 * u_perp - C_perp * a_perp) / m) - (v_para * phi_dot)
    phi_ddot   = (km4 * u_phi / I) - (C_phi * phi_dot / I) + (km2 * u_para / I)

    x_dot   = v_para * np.cos(phi) - v_perp * np.sin(phi)
    y_dot   = v_para * np.sin(phi) + v_perp * np.cos(phi)
    z_dot   = 0*x
    w_dot   = 0*x
    zeta_dot = 0*x
    m_dot   = 0*x
    I_dot   = 0*x
    C_para_dot = 0*x
    C_perp_dot = 0*x
    C_phi_dot  = 0*x
    km1 = 0*x
    km2 = 0*x
    km3 = 0*x
    km4 = 0*x

    return [x_dot, y_dot, z_dot, v_para_dot, v_perp_dot, phi_dot, phi_ddot,
            w_dot, zeta_dot, m_dot, I_dot, C_para_dot, C_perp_dot, C_phi_dot,
            km1, km2, km3, km4]


def h(X, U):
    x, y, z, v_para, v_perp, phi, phi_dot, w, zeta, m, I, C_para, C_perp, C_phi, km1, km2, km3, km4 = X
    u_para, u_perp, u_phi = U

    a_para = v_para - w * np.cos(phi - zeta)
    a_perp = v_perp + w * np.sin(phi - zeta)
    a      = np.sqrt(a_para**2 + a_perp**2)
    gamma  = np.arctan2(a_perp, a_para)

    g   = np.sqrt(v_para**2 + v_perp**2)
    psi = np.arctan2(v_perp, v_para)
    r   = g / z

    if np.array(phi).ndim > 0:
        if np.array(phi).shape[0] > 1:
            phi   = np.unwrap(phi)
            psi   = np.unwrap(psi)
            gamma = np.unwrap(gamma)

    return [phi, psi, gamma, a, g, r]

## JAX dynamics and measurement (for `JaxSimulator`)

These are the **only new functions you need to write** compared to the original notebook.

**Two differences from the legacy functions:**

1. Use `jnp.*` for all math (`jnp.cos`, `jnp.sin`, `jnp.arctan2`, `jnp.sqrt`) and return `jnp.array(...)` instead of a list.

2. **Remove the angle-unwrapping block** from `h_jax`. The original `h` checks `phi.ndim > 0` to unwrap angles when given a whole trajectory, but `JaxSimulator` calls `h(x, u)` at each individual time step where `x` is a 1-D array of scalar values — so unwrapping is unnecessary and the data-dependent `if` would break JAX tracing.

In [ ]:
def f_jax(x, u):
    # Unpack states (same order as state_names)
    (x_pos, y_pos, z, v_para, v_perp, phi, phi_dot,
     w, zeta, m, I, C_para, C_perp, C_phi,
     km1, km2, km3, km4) = [x[i] for i in range(18)]
    u_para, u_perp, u_phi = u[0], u[1], u[2]

    # Air velocity
    a_para = v_para - w * jnp.cos(phi - zeta)
    a_perp = v_perp + w * jnp.sin(phi - zeta)

    # Accelerations
    v_para_dot = ((km1 * u_para - C_para * a_para) / m) + (v_perp * phi_dot)
    v_perp_dot = ((km3 * u_perp - C_perp * a_perp) / m) - (v_para * phi_dot)
    phi_ddot   = (km4 * u_phi / I) - (C_phi * phi_dot / I) + (km2 * u_para / I)

    # Position derivatives
    x_dot = v_para * jnp.cos(phi) - v_perp * jnp.sin(phi)
    y_dot = v_para * jnp.sin(phi) + v_perp * jnp.cos(phi)

    # Constant states have zero derivative
    return jnp.array([
        x_dot, y_dot, 0.0, v_para_dot, v_perp_dot, phi_dot, phi_ddot,
        0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
    ])


def h_jax(x, u):
    # Unpack states
    (x_pos, y_pos, z, v_para, v_perp, phi, phi_dot,
     w, zeta, m, I, C_para, C_perp, C_phi,
     km1, km2, km3, km4) = [x[i] for i in range(18)]

    # Air velocity
    a_para = v_para - w * jnp.cos(phi - zeta)
    a_perp = v_perp + w * jnp.sin(phi - zeta)
    a      = jnp.sqrt(a_para**2 + a_perp**2)
    gamma  = jnp.arctan2(a_perp, a_para)

    # Ground speed
    g   = jnp.sqrt(v_para**2 + v_perp**2)
    psi = jnp.arctan2(v_perp, v_para)

    # Optic flow
    r = g / z

    # No angle unwrapping — JAX evaluates h(x, u) one time step at a time
    return jnp.array([phi, psi, gamma, a, g, r])

# Create simulator objects

* **`Simulator`** (do_mpc / CasADi) — used for MPC and trajectory integration. **Unchanged.**
* **`JaxSimulator`** — pure JAX forward integrator used only for the observability computation.

In [ ]:
# Legacy simulator — unchanged
simulator = Simulator(f, h, dt=dt,
                      state_names=state_names,
                      input_names=input_names,
                      measurement_names=measurement_names)

# JAX simulator — same parameters, JAX-compatible functions
jax_simulator = JaxSimulator(f_jax, h_jax, dt=dt,
                             state_names=state_names,
                             input_names=input_names,
                             measurement_names=measurement_names,
                             integrator='rk4')

# Set up model predictive control

In [ ]:
# Parameters in SI units
m_val   = 0.25e-6   # [kg]
I_val   = 5.2e-13   # [N*m*s^2]
C_phi_val  = 27.36e-12  # [N*m*s]
C_para_val = m_val / 0.170
C_perp_val = C_para_val

# Scale to mg / mm units
m_val      = m_val      * 1e6
I_val      = I_val      * 1e6 * (1e3)**2
C_phi_val  = C_phi_val  * 1e6 * (1e3)**2
C_para_val = C_para_val * 1e6
C_perp_val = C_perp_val * 1e6

In [ ]:
tsim = np.arange(0, 0.4, step=dt)

setpoint = {
    'x':      0.0  * np.ones_like(tsim),
    'y':      0.0  * np.ones_like(tsim),
    'z':      0.2  * np.ones_like(tsim),
    'v_para': 0.3  * np.ones_like(tsim) + 0.01 * tsim,
    'v_perp': 0.01 * np.ones_like(tsim) + 0.01 * tsim,
    'phi':    (np.pi / 4) * np.ones_like(tsim),
    'phi_dot': 0.0 * np.ones_like(tsim),
    'w':      0.4  * np.ones_like(tsim),
    'zeta':   np.pi * np.ones_like(tsim),
    'm':      m_val      * np.ones_like(tsim),
    'I':      I_val      * np.ones_like(tsim),
    'C_para': C_para_val * np.ones_like(tsim),
    'C_perp': C_perp_val * np.ones_like(tsim),
    'C_phi':  C_phi_val  * np.ones_like(tsim),
    'km1': 1.0 * np.ones_like(tsim),
    'km2': 0.0 * np.ones_like(tsim),
    'km3': 1.0 * np.ones_like(tsim),
    'km4': 1.0 * np.ones_like(tsim),
}

# Add a turn
setpoint['phi'][20:] = setpoint['phi'][20] - np.pi / 2

simulator.update_dict(setpoint, name='setpoint')

In [ ]:
cost = ((simulator.model.x['v_para'] - simulator.model.tvp['v_para_set'])**2 +
        (simulator.model.x['v_perp'] - simulator.model.tvp['v_perp_set'])**2 +
        (simulator.model.x['phi']    - simulator.model.tvp['phi_set'])**2)

simulator.mpc.set_objective(mterm=cost, lterm=cost)
simulator.mpc.set_rterm(u_para=1e-6, u_perp=1e-6, u_phi=1e-6)

# Run model predictive control

In [ ]:
st = time.time()
t_sim, x_sim, u_sim, y_sim = simulator.simulate(x0=None, mpc=True, return_full_output=True)
print('MPC elapsed time:', time.time() - st, 's')

In [ ]:
simulator.plot('setpoint')

In [ ]:
simulator.plot('u')

# Re-run simulator with MPC inputs (open-loop replay)

Replays the MPC inputs in open-loop to get the same state trajectory.
The resulting `(t_sim, x_sim, u_sim)` is what gets passed to the JAX
observability class.

In [ ]:
t_sim, x_sim, u_sim, y_sim = simulator.simulate(x0=None, mpc=False, u=u_sim, return_full_output=True)
simulator.plot('x')

# Observability

## Construct sliding observability matrix — JAX version

This is the **one line that changes** relative to the original notebook:

```python
# Original (numerical perturbations — slow)
SEOM = SlidingEmpiricalObservabilityMatrix(simulator, t_sim, x_sim, u_sim, w=w, eps=1e-4)

# JAX (exact Jacobian via vmap + jacfwd — fast)
SEOM = JaxSlidingEmpiricalObservabilityMatrix(jax_simulator, t_sim, x_sim, u_sim, w=w)
```

The output `SEOM.O_df_sliding` has the **same format** as before, so all
downstream code is completely unchanged.

In [ ]:
w = 4  # window size in time-steps

st = time.time()
SEOM = JaxSlidingEmpiricalObservabilityMatrix(jax_simulator, t_sim, x_sim, u_sim, w=w)
print(f'JAX sliding observability: {time.time() - st:.2f} s  ({len(SEOM.O_df_sliding)} windows)')

In [ ]:
O_sliding = SEOM.get_observability_matrix()
print(len(O_sliding), 'windows')

In [ ]:
# Visualize first sliding observability matrix
OI = ObservabilityMatrixImage(O_sliding[0], cmap='bwr')
OI.plot(scale=1.0)

## Compute Fisher information matrix & inverse for first window

Identical to the original notebook.

In [ ]:
R = {'phi': 0.1, 'psi': 0.1, 'gamma': 0.1, 'a': 0.1, 'g': 0.1, 'r': 0.1}

In [ ]:
FO = FisherObservability(SEOM.O_df_sliding[0], R=R, lam=1e-6)
F, F_inv, R_matrix = FO.get_fisher_information()
F_inv

## Compute Fisher information & inverse for each sliding window

Identical to the original notebook.

In [ ]:
o_sensors    = ['phi', 'psi', 'gamma']
o_states     = ['v_para', 'v_perp', 'phi', 'phi_dot', 'w', 'zeta',
                'z', 'm', 'I', 'C_para', 'C_perp', 'C_phi']
o_time_steps = np.arange(0, w)

SFO = SlidingFisherObservability(SEOM.O_df_sliding, time=SEOM.t_sim, lam=1e-6, R=R,
                                 states=o_states, sensors=o_sensors,
                                 time_steps=o_time_steps, w=None)

In [ ]:
EV_aligned = SFO.get_minimum_error_variance()
EV_aligned

## Visualize observability matrix subset

In [ ]:
OI = ObservabilityMatrixImage(SFO.FO[-1].O)
OI.plot()

# Plot error variance as color on state time-series

Identical to the original notebook.

In [ ]:
try:
    EV_no_nan = EV_aligned.fillna(method='bfill').fillna(method='ffill')
except:
    EV_no_nan = EV_aligned.bfill().ffill()

In [ ]:
states  = list(SFO.FO[0].O.columns)
n_state = len(states)

fig, ax = plt.subplots(n_state, 2, figsize=(6, n_state * 2), dpi=150)
ax = np.atleast_2d(ax)

cmap = 'inferno_r'
min_ev = np.min(EV_no_nan.iloc[:, 2:].values)
max_ev = np.max(EV_no_nan.iloc[:, 2:].values)
log_tick_high = int(np.ceil(np.log10(max_ev)))
log_tick_low  = int(np.floor(np.log10(min_ev)))
cnorm = mpl.colors.LogNorm(10**log_tick_low, 10**log_tick_high)

for n, state_name in enumerate(states):
    colorline(x_sim['x'], x_sim['y'], EV_no_nan[state_name].values,
              ax=ax[n, 0], cmap=cmap, norm=cnorm)
    colorline(t_sim, EV_no_nan[state_name].values, EV_no_nan[state_name].values,
              ax=ax[n, 1], cmap=cmap, norm=cnorm)

    cax  = ax[n, -1].inset_axes([1.03, 0.0, 0.04, 1.0])
    cbar = fig.colorbar(
        mpl.cm.ScalarMappable(norm=cnorm, cmap=cmap), cax=cax,
        ticks=np.logspace(log_tick_low, log_tick_high, log_tick_high - log_tick_low + 1))
    cbar.set_label('min. EV: ' + state_name, rotation=270, fontsize=7, labelpad=8)
    cbar.ax.tick_params(labelsize=6)

    ax[n, 0].set_ylim(np.min(x_sim['y']) - 0.01, np.max(x_sim['y']) + 0.01)
    ax[n, 0].set_xlim(np.min(x_sim['x']) - 0.01, np.max(x_sim['x']) + 0.01)
    ax[n, 0].set_ylabel('y', fontsize=7)
    ax[n, 0].set_xlabel('x', fontsize=7)
    ax[n, 0].set_aspect(1.0)

    ax[n, 1].set_ylim(10**log_tick_low, 10**log_tick_high)
    ax[n, 1].set_yscale('log')
    ax[n, 1].set_ylabel('min. EV: ' + state_name, fontsize=7)
    ax[n, 1].set_yticks(
        np.logspace(log_tick_low, log_tick_high, log_tick_high - log_tick_low + 1))

for a in ax.flat:
    a.tick_params(axis='both', labelsize=6)
for a in ax[:, 1]:
    a.set_xlabel('time (s)', fontsize=7)
    a.set_xlim(-0.1, t_sim[-1] + 0.1)

fig.subplots_adjust(wspace=0.3, hspace=0.4)
plt.show()